# Smart Grid Analytics - End-to-End Project Requirements

## Project Overview

You are a data engineer working for a smart grid utility company. Your task is to build a complete data pipeline that processes raw smart meter data, weather information, customer accounts, and grid events to create business intelligence dashboards.

The project follows the **Medallion Architecture** pattern:
- **🥉 Bronze**: Raw ingestion layer (provided)
- **🥈 Silver**: Cleaned and conformed data layer (your task)
- **🥇 Gold**: Business intelligence and analytics layer (your task)

**Technology Stack:**
- Databricks (Free Edition - Unity Catalog & Volumes)
- PySpark
- Delta Lake
- Auto Loader (for streaming/batch ingestion)

## 🥉 Bronze Layer (Provided)

The bronze layer contains raw data files in JSON format with **intentional data quality issues**. These are stored in Unity Catalog Volumes.

### Bronze Tables:

1. **`raw_meter_readings`** (52,000+ rows - Main Table)
   - `id`: Unique reading identifier
   - `meter_id`: Reference to smart meter
   - `kwh`: Energy consumption in kilowatt-hours
   - `voltage`: Voltage reading
   - `timestamp`: When the reading was taken
   - `status_code`: Meter status (active, inactive, error, etc.)

2. **`raw_weather_station`** (~520 rows)
   - `station_id`: Weather station identifier
   - `temp`: Temperature in Fahrenheit
   - `humidity`: Humidity percentage (0-100)
   - `wind_speed`: Wind speed in mph
   - `recorded_at`: Timestamp of weather reading
   - `lat`: Latitude coordinate
   - `lon`: Longitude coordinate

3. **`raw_customer_accounts`** (~2,050 rows)
   - `account_id`: Unique customer account identifier
   - `name`: Customer name
   - `address`: Street address
   - `zip_code`: ZIP code
   - `service_type`: Type of service (residential, commercial, industrial)
   - `created_at`: Account creation timestamp

4. **`raw_smart_meters`** (~1,530 rows)
   - `meter_id`: Unique meter identifier
   - `account_id`: Reference to customer account
   - `model`: Meter model number
   - `installation_date`: When meter was installed
   - `fw_version`: Firmware version

5. **`raw_grid_events`** (~3,050 rows)
   - `event_id`: Unique event identifier
   - `timestamp`: When event occurred
   - `event_type`: Type of event (voltage_spike, power_outage, etc.)
   - `severity`: Event severity (low, medium, high, critical)
   - `description_json`: JSON string with additional event details

**⚠️ Data Quality Issues to Handle:**
- Missing values (NULL)
- Duplicate records
- Invalid date/timestamp formats
- Out-of-range values (negative kWh, invalid coordinates)
- Inconsistent case (mixed uppercase/lowercase)
- Orphaned records (references to non-existent entities)
- Malformed JSON in `description_json` field
- Data type issues

## 🥈 Silver Layer (Your Task)

Create cleaned and conformed tables in the `smart_grid_project.silver_schema` schema.

### Silver Tables to Create:

1. **`dim_customer`** (SCD Type 2 - Slowly Changing Dimension)
   - Track customer address changes over time
   - Include: `account_id`, `name`, `address`, `zip_code`, `service_type`, `valid_from`, `valid_to`, `is_current`
   - Requirements:
     - Clean and standardize customer data from `raw_customer_accounts`
     - Handle duplicates (keep most recent)
     - Standardize `service_type` to lowercase
     - Validate ZIP codes (5 digits)
     - Implement SCD Type 2 logic for address changes

2. **`dim_meter`**
   - Standardized meter information
   - Include: `meter_id`, `account_id`, `model`, `installation_date`, `fw_version_major`, `fw_version_minor`, `fw_version_patch`
   - Requirements:
     - Clean meter data from `raw_smart_meters`
     - Parse firmware version (e.g., "v2.3.1" → major=2, minor=3, patch=1)
     - Standardize model names (handle case inconsistencies)
     - Validate installation dates
     - Handle orphaned records (meters without valid accounts)

3. **`fact_readings`**
   - Cleansed meter readings with enriched data
   - Include: `reading_id`, `meter_id`, `account_id`, `kwh`, `voltage`, `timestamp`, `status_code`, `weather_station_id`, `temp`, `humidity`, `wind_speed`, `zip_code`
   - Requirements:
     - Clean readings from `raw_meter_readings`
     - Remove duplicates
     - Filter invalid values (negative kWh, out-of-range voltage)
     - Join with `dim_meter` to get `account_id`
     - Join with weather data based on **closest lat/lon** (spatial join)
     - Join with customer to get `zip_code`
     - Handle missing timestamps
     - Standardize `status_code` to lowercase

4. **`dim_geography`**
   - Standardized ZIP code to city/region mapping
   - Include: `zip_code`, `city`, `state`, `region`
   - Requirements:
     - Create from customer data
     - Deduplicate ZIP codes
     - You may need to infer or create city/state mappings (can use lookup or simple logic)

**Key Techniques to Use:**
- **Auto Loader** with schema evolution for reading JSON files
- **Rescued data column** to capture malformed records
- **PySpark functions**: `filter()`, `when()`, `regexp_replace()`, `to_date()`, `to_timestamp()`, `upper()`, `lower()`, `trim()`
- **Data quality checks**: Validate ranges, formats, and relationships
- **Deduplication**: Use `dropDuplicates()` or window functions
- **Spatial joins**: Calculate distance between meter locations and weather stations (you may need to infer meter locations from customer ZIP codes)

## 🥇 Gold Layer (Your Task)

Create business intelligence tables in the `smart_grid_project.gold_schema` schema.

### Gold Tables to Create:

1. **`kpi_peak_demand_ratio`**
   - Calculate Peak Demand Ratio per neighborhood (ZIP code)
   - Formula: `Peak Hour Usage / Average Daily Usage`
   - Include: `zip_code`, `date`, `peak_hour_usage`, `avg_daily_usage`, `peak_demand_ratio`
   - Requirements:
     - Group readings by ZIP code and date
     - Identify peak hour (hour with highest total kWh)
     - Calculate average daily usage
     - Compute ratio

2. **`kpi_grid_stability_index`**
   - Measure grid stability based on events and voltage fluctuations
   - Formula: `Count of High-Severity Events / Count of Voltage Fluctuations`
   - Include: `date`, `high_severity_events`, `voltage_fluctuations`, `stability_index`
   - Requirements:
     - Count high-severity grid events (severity = 'high' or 'critical')
     - Count voltage fluctuations (readings where voltage deviates significantly from normal range, e.g., < 100V or > 250V)
     - Calculate index (handle division by zero)

3. **`kpi_climate_impact_factor`**
   - Correlation between temperature and household consumption
   - Formula: `Average kWh per 1°C temperature increase`
   - Include: `zip_code`, `temp_range`, `avg_kwh`, `temp_increase`, `kwh_increase`, `impact_factor`
   - Requirements:
     - Group readings by ZIP code and temperature ranges
     - Calculate average consumption per temperature range
     - Compute correlation/impact factor (how much kWh increases per 1°C)
     - You can use simple linear approximation or more sophisticated methods

**Key Techniques to Use:**
- **Aggregations**: `groupBy()`, `agg()`, `sum()`, `avg()`, `max()`, `min()`, `count()`
- **Window functions**: For time-based calculations
- **Joins**: Combine fact and dimension tables
- **Date/time functions**: Extract hour, day, month from timestamps
- **Statistical functions**: For correlation calculations

## Implementation Guidelines

### Step 1: Environment Setup
1. Run the `01_project_data_setup.ipynb` notebook to create bronze data
2. Create schemas for silver and gold layers:
   ```sql
   CREATE SCHEMA IF NOT EXISTS smart_grid_project.silver_schema;
   CREATE SCHEMA IF NOT EXISTS smart_grid_project.gold_schema;
   ```

### Step 2: Bronze to Silver Pipeline
1. Use **Auto Loader** to read JSON files from the volume
2. Enable **schema evolution** and **rescued data column**
3. Clean and transform each bronze table
4. Write to Delta tables in silver schema

### Step 3: Silver to Gold Pipeline
1. Read from silver tables
2. Perform aggregations and calculations
3. Write to Delta tables in gold schema

### Step 4: Data Quality Checks
1. Validate record counts
2. Check for NULLs in critical fields
3. Verify relationships (foreign keys)
4. Validate calculated metrics (ranges, formats)

### Code Structure Example:
```python
# Read bronze data with Auto Loader
bronze_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/path/to/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load("/Volumes/smart_grid_project/bronze_schema/raw/meter_readings"))

# Clean and transform
silver_df = (bronze_df
    .filter(col("kwh").isNotNull() & (col("kwh") > 0))
    .withColumn("status_code", lower(col("status_code")))
    .dropDuplicates(["id"]))

# Write to Delta
(silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/path/to/checkpoint")
    .table("smart_grid_project.silver_schema.fact_readings"))
```

## Deliverables

1. **Silver Layer Tables** (4 tables):
   - `dim_customer` (SCD Type 2)
   - `dim_meter`
   - `fact_readings`
   - `dim_geography`

2. **Gold Layer Tables** (3 tables):
   - `kpi_peak_demand_ratio`
   - `kpi_grid_stability_index`
   - `kpi_climate_impact_factor`

3. **Documentation**:
   - Explain your data cleaning approach
   - Document any assumptions made
   - Describe how you handled data quality issues
   - Explain your KPI calculation logic

4. **Data Quality Report**:
   - Number of records processed
   - Number of records filtered/dropped
   - Number of rescued records
   - Data quality metrics

## Evaluation Criteria

1. **Data Quality** (40%):
   - Proper handling of missing values
   - Duplicate removal
   - Data type conversions
   - Validation of ranges and formats
   - Use of rescued data column

2. **Data Modeling** (30%):
   - Correct SCD Type 2 implementation
   - Proper dimension and fact table design
   - Appropriate joins and relationships
   - Spatial join logic (weather to readings)

3. **Business Logic** (20%):
   - Correct KPI calculations
   - Accurate aggregations
   - Proper date/time handling

4. **Code Quality** (10%):
   - Clean, readable code
   - Proper use of PySpark functions
   - Efficient transformations
   - Comments and documentation

## Tips and Best Practices

1. **Start with Bronze to Silver**: Focus on cleaning one table at a time
2. **Test Incrementally**: Verify each transformation step
3. **Handle Edge Cases**: NULL values, empty strings, extreme values
4. **Use Rescued Data**: Check `_rescued_data` column for malformed records
5. **Optimize Joins**: Use broadcast joins for small dimension tables
6. **Partition Strategically**: Partition fact tables by date for better performance
7. **Document Assumptions**: Note any business logic decisions
8. **Validate Results**: Check record counts and sample data after each step

## Resources

- [Databricks Auto Loader Documentation](https://docs.databricks.com/ingestion/auto-loader/index.html)
- [Delta Lake Documentation](https://docs.delta.io/)
- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)

---

